In [9]:
import os
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import UnexpectedAlertPresentException, NoAlertPresentException
from webdriver_manager.chrome import ChromeDriverManager

In [10]:
# 1. 설정
TARGET_URL = "https://defense.na.go.kr:444/cmmit/bbs/BCMT2003/list.do?pageIndex=1&menuNo=2000031&pageUnit=30"
SAVE_DIR = os.path.abspath("defense_audit_files")

if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

chrome_options = Options()
prefs = {
    "download.default_directory": SAVE_DIR,
    "download.prompt_for_download": False,
    "directory_upgrade": True,
    "safebrowsing.enabled": False
}
chrome_options.add_experimental_option("prefs", prefs)
chrome_options.add_argument('--ignore-certificate-errors')
chrome_options.add_argument('--ignore-ssl-errors')

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

def start_download():
    try:
        driver.get(TARGET_URL)
        wait = WebDriverWait(driver, 15)
        
        # 게시판 테이블 로딩 대기
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.board01 table tbody tr")))
        
        # [수정] 실제 데이터가 있는 행의 '인덱스'를 미리 확보합니다.
        all_rows = driver.find_elements(By.CSS_SELECTOR, "div.board01 table tbody tr")
        data_indices = []
        for idx, r in enumerate(all_rows):
            # td가 1개인 행(공지/데이터없음)은 제외하고 실제 게시글만 인덱스 저장
            if len(r.find_elements(By.TAG_NAME, "td")) > 1:
                data_indices.append(idx)
        
        print(f"실제 게시글 개수: {len(data_indices)}개")

        for i, row_idx in enumerate(data_indices):
            try:
                # 매번 요소를 새로 찾아서 'row_idx'로 접근해야 밀리지 않습니다.
                current_rows = driver.find_elements(By.CSS_SELECTOR, "div.board01 table tbody tr")
                row = current_rows[row_idx]
                
                # [수정] 제목 추출 방식을 클래스명 대신 태그 위치로 변경 (에러 방지용)
                try:
                    cells = row.find_elements(By.TAG_NAME, "td")
                    # 보통 2~3번째 칸이 제목입니다.
                    title_text = cells[1].text if len(cells) > 1 else "Unknown Title"
                except:
                    title_text = "제목 추출 실패"
                
                print(f"[{i+1}/{len(data_indices)}] 처리 중: {title_text}")

                download_btns = row.find_elements(By.CSS_SELECTOR, "a.btn_board_download")
                if not download_btns:
                    print(f"   -> 다운로드 버튼 없음")
                    continue

                # 레이어 열기
                driver.execute_script("arguments[0].click();", download_btns[0])
                time.sleep(1.2)

                # 파일 링크 추출 및 중복 제거
                raw_links = row.find_elements(By.CSS_SELECTOR, ".board_download_layer ul li a[onclick*='Download']")
                unique_links = {}
                for link in raw_links:
                    onclick_val = link.get_attribute("onclick")
                    if onclick_val not in unique_links:
                        unique_links[onclick_val] = (link, link.get_attribute("title"))

                if not unique_links:
                    print(f"   -> [참고] 레이어 내 파일 링크가 비어있음")

                for onclick_func, (link_obj, f_name) in unique_links.items():
                    try:
                        print(f"   -> 다운로드 실행: {f_name}")
                        driver.execute_script("arguments[0].click();", link_obj)
                        time.sleep(2) 

                        # [핵심] 알림창(파일 없음 등)이 뜨면 즉시 닫기
                        try:
                            alert = driver.switch_to.alert
                            print(f"      ⚠️ 서버 알림: {alert.text}")
                            alert.accept()
                        except NoAlertPresentException:
                            pass

                    except UnexpectedAlertPresentException:
                        try:
                            alert = driver.switch_to.alert
                            alert.accept()
                        except: pass
                        continue

            except Exception as e:
                print(f"[{i+1}번 행] 에러 발생 (건너뜀 방지 처리): {e}")
                continue

        print(f"\n✅ 모든 게시글 확인 완료! 폴더: {SAVE_DIR}")

    finally:
        driver.quit()

if __name__ == "__main__":
    start_download()

실제 게시글 개수: 26개
[1/26] 처리 중: 
   -> 다운로드 실행: 2024년도 국정감사 결과보고서(국방위원회).hwp
   -> 다운로드 실행: 2024년도 국정감사 결과보고서(국방위원회).pdf
[2/26] 처리 중: 
   -> 다운로드 실행: 2022년도 국방위원회 국정감사 결과보고서.hwp
   -> 다운로드 실행: 2022년도 국방위원회 국정감사 결과보고서.pdf
[3/26] 처리 중: 
   -> 다운로드 실행: 2020년도 국정감사결과보고서.hwp
   -> 다운로드 실행: 2020년도 국정감사결과보고서.pdf
[4/26] 처리 중: 
   -> 다운로드 실행: 2018년도 국정감사 결과보고서.hwp
   -> 다운로드 실행: 2018년도 국정감사 결과보고서.pdf
[5/26] 처리 중: 
   -> 다운로드 실행: 2017_국방위원회_국정감사 결과보고서(최종본).hwp
   -> 다운로드 실행: 2017_국방위원회_국정감사 결과보고서(최종본).pdf
[6/26] 처리 중: 
   -> 다운로드 실행: 2016년도 국정감사 결과보고서(최종).hwp
      ⚠️ 서버 알림: [2016년도 국정감사 결과보고서(최종).hwp] 파일이 존재하지 않습니다.
   -> 다운로드 실행: 2016년도 국정감사 결과보고서(최종).pdf
[6번 행] 에러 발생 (건너뜀 방지 처리): Message: stale element reference: stale element not found
  (Session info: chrome=143.0.7499.148); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#staleelementreferenceexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0xe012d